# Pandas DataFrame Basics & Selection

**Instructor:** Netra Prasad Neupane

This notebook is the hands-on companion to today's class. We'll build a small student-scores
dataset from scratch and use it to practice every idea from the slides: Series and DataFrames,
loading data, inspecting it, selecting rows and columns, boolean filtering, sorting, renaming,
adding columns, type conversion, and dates.

Run the cells in order — later sections reuse the dataset created in Section 3.

## Setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
print("pandas version:", pd.__version__)

pandas version: 2.2.3


## 1. Series — a single labeled column

A **Series** is a one-dimensional array where every value has a label (an *index*). Think of it
as one column from a spreadsheet, lifted out on its own — you still know which row each value
belongs to, because of the label next to it.

Let's make a Series of exam marks for five students, indexed by their names.

In [2]:
marks = pd.Series([88, 72, 45, 91, 67], index=["Aarav", "Bibek", "Chandni", "Deepika", "Esha"])
marks

Aarav      88
Bibek      72
Chandni    45
Deepika    91
Esha       67
dtype: int64

In [3]:
# Every Series has two parts: the labels (index) and the data (values)
print("Index :", marks.index.tolist())
print("Values:", marks.values)
print("Bibek's mark:", marks["Bibek"])

Index : ['Aarav', 'Bibek', 'Chandni', 'Deepika', 'Esha']
Values: [88 72 45 91 67]
Bibek's mark: 72


## 2. DataFrame — a table made of Series

A **DataFrame** is a two-dimensional table: many columns (each one a Series) that all share the
same row labels. It behaves like a full spreadsheet — rows, columns, and a grid of values.

Let's build a tiny DataFrame by hand before we load a real dataset.

In [4]:
mini = pd.DataFrame({
    "name": ["Aarav", "Bibek", "Chandni", "Deepika", "Esha"],
    "subject": ["Python Programming", "Statistics", "Python Programming", "Machine Learning", "Statistics"],
    "marks": [88, 72, 45, 91, 67],
})
mini

,name,subject,marks
0,Aarav,Python Programming,88
1,Bibek,Statistics,72
2,Chandni,Python Programming,45
3,Deepika,Machine Learning,91
4,Esha,Statistics,67


Notice three things every DataFrame has:

- **Index** — the row labels on the left (here, just 0-4, auto-generated)
- **Columns** — the column names across the top
- **Values** — the actual data in the grid

```
mini.index    -> row labels
mini.columns  -> column names
mini.values   -> the raw data as a 2D array
```

In [5]:
print("Index  :", mini.index.tolist())
print("Columns:", mini.columns.tolist())
print("Shape  :", mini.shape, "-> (rows, columns)")

Index  : [0, 1, 2, 3, 4]
Columns: ['name', 'subject', 'marks']
Shape  : (5, 3) -> (rows, columns)


## 3. Creating a DataFrame: dictionary and other ways

The `mini` DataFrame above was built from a **dictionary of lists** — the most common way. But
pandas accepts several other shapes of input too. It helps to recognize them, since real data
doesn't always arrive as a tidy dictionary.

In [6]:
# 1. Dictionary of lists — each key becomes a column
pd.DataFrame({
    "name": ["Aarav", "Bibek"],
    "marks": [88, 72],
})

,name,marks
0,Aarav,88
1,Bibek,72


In [7]:
# 2. List of dictionaries — each dictionary becomes one row
pd.DataFrame([
    {"name": "Aarav", "marks": 88},
    {"name": "Bibek", "marks": 72},
])

,name,marks
0,Aarav,88
1,Bibek,72


In [8]:
# 3. List of lists (or a 2D array) — plain rows of values, with column names given separately
pd.DataFrame(
    [["Aarav", 88], ["Bibek", 72]],
    columns=["name", "marks"],
)

,name,marks
0,Aarav,88
1,Bibek,72


All three produce the same table. Which one you reach for usually depends on how the data
already looks in your code — a dictionary of lists if you're building columns, a list of
dictionaries if you're building rows one at a time (e.g. from an API response).

In [9]:
import numpy as np
x = np.array([[0.9], [0.8], [0.7]])

In [10]:
print(x)

[[0.9]
 [0.8]
 [0.7]]


In [11]:
x.shape

(3, 1)

## 4. Loading data (CSV / Excel)

In real work you rarely type a DataFrame by hand — you load one from a file. Let's create a
slightly bigger dataset (20 students, 5 subjects) and save it as both CSV and Excel, so we can
practice loading each format the way you would with real files.

In [12]:
data = {
    "student_id": list(range(101, 121)),
    "name": [
        "Aarav Sharma","Bibek Thapa","Chandni Rai","Deepika Gurung","Esha KC",
        "Faisal Khan","Gita Adhikari","Hari Bahadur","Isha Shrestha","Jeevan Lama",
        "Kabita Basnet","Lokesh Poudel","Maya Tamang","Nabin Karki","Oshin Magar",
        "Prakash Yadav","Queenie Rana","Rohit Bista","Sabina Ghale","Tenzin Sherpa"
    ],
    "city": [
        "Kathmandu","Pokhara","Biratnagar","Kathmandu","Lalitpur",
        "Butwal","Pokhara","Kathmandu","Lalitpur","Biratnagar",
        "Kathmandu","Pokhara","Butwal","Kathmandu","Lalitpur",
        "Biratnagar","Pokhara","Kathmandu","Butwal","Lalitpur"
    ],
    "subject": [
        "Python Programming","Statistics","Python Programming","Machine Learning","Statistics",
        "Python Programming","Machine Learning","SQL","Statistics","Python Programming",
        "SQL","Machine Learning","Python Programming","Statistics","SQL",
        "Machine Learning","Python Programming","Statistics","SQL","Machine Learning"
    ],
    "marks": [88, 72, 45, 91, 67, 58, 39, 82, 95, 61, 74, 48, 90, 55, 68, 77, 33, 85, 62, 93],
    "attendance_percent": [95.0, 88.5, 70.0, 98.0, 80.0, 65.0, 55.0, 90.0, 99.0, 72.0,
                            85.0, 60.0, 96.0, 75.0, 82.0, 89.0, 50.0, 92.0, 78.0, 97.0],
    "exam_date": [
        "2026-05-02","2026-05-03","2026-05-02","2026-05-04","2026-05-03",
        "2026-05-02","2026-05-04","2026-05-05","2026-05-03","2026-05-02",
        "2026-05-05","2026-05-04","2026-05-02","2026-05-03","2026-05-05",
        "2026-05-04","2026-05-02","2026-05-03","2026-05-05","2026-05-04"
    ],
}

seed_df = pd.DataFrame(data)
seed_df.to_csv("class_scores.csv", index=False)
seed_df.to_excel("class_scores.xlsx", index=False, sheet_name="Midterm")
print("Files written: class_scores.csv, class_scores.xlsx")

Files written: class_scores.csv, class_scores.xlsx


In [13]:
# Loading from CSV — the most common case
df = pd.read_csv("class_scores.csv")
df.head()

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
1,102,Bibek Thapa,Pokhara,Statistics,72,88.5,2026-05-03
2,103,Chandni Rai,Biratnagar,Python Programming,45,70.0,2026-05-02
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
4,105,Esha KC,Lalitpur,Statistics,67,80.0,2026-05-03


In [14]:
# Loading from Excel is almost identical — you just point at the sheet name
df_excel = pd.read_excel("class_scores.xlsx", sheet_name="Midterm")
df_excel.head()

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
1,102,Bibek Thapa,Pokhara,Statistics,72,88.5,2026-05-03
2,103,Chandni Rai,Biratnagar,Python Programming,45,70.0,2026-05-02
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
4,105,Esha KC,Lalitpur,Statistics,67,80.0,2026-05-03


A few `read_csv` / `read_excel` options worth knowing:

- `sep=","` — change the delimiter (e.g. `sep=";"` for semicolon-separated files)
- `header=0` — which row holds the column names (use `header=None` if there isn't one)
- `index_col="student_id"` — use an existing column as the row index instead of 0,1,2,...
- `sheet_name="Midterm"` — which sheet to read from an Excel workbook

From here on, we'll work with `df` (loaded from the CSV).

## 5. Inspecting the data

Before doing anything with a new dataset, always look at its shape, structure, and summary
statistics. This catches surprises early — wrong types, missing values, unexpected ranges.

In [15]:
df.head()      # first 5 rows

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
1,102,Bibek Thapa,Pokhara,Statistics,72,88.5,2026-05-03
2,103,Chandni Rai,Biratnagar,Python Programming,45,70.0,2026-05-02
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
4,105,Esha KC,Lalitpur,Statistics,67,80.0,2026-05-03


In [16]:
df.tail(3)     # last 3 rows

,student_id,name,city,subject,marks,attendance_percent,exam_date
17,118,Rohit Bista,Kathmandu,Statistics,85,92.0,2026-05-03
18,119,Sabina Ghale,Butwal,SQL,62,78.0,2026-05-05
19,120,Tenzin Sherpa,Lalitpur,Machine Learning,93,97.0,2026-05-04


In [17]:
df.shape       # (rows, columns)

(20, 7)

In [18]:
df.info()      # column names, non-null counts, and dtypes in one view

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          20 non-null     int64  
 1   name                20 non-null     object 
 2   city                20 non-null     object 
 3   subject             20 non-null     object 
 4   marks               20 non-null     int64  
 5   attendance_percent  20 non-null     float64
 6   exam_date           20 non-null     object 
dtypes: float64(1), int64(2), object(4)
memory usage: 1.2+ KB


In [19]:
df.dtypes      # just the data type of each column

student_id              int64
name                   object
city                   object
subject                object
marks                   int64
attendance_percent    float64
exam_date              object
dtype: object

In [20]:
df.describe()               # summary stats for numeric columns

,student_id,marks,attendance_percent
count,20.00000,20.000000,20.000000
mean,110.50000,69.150000,80.825000
std,5.91608,18.790465,14.886655
min,101.00000,33.000000,50.000000
25%,105.75000,57.250000,71.500000
50%,110.50000,70.000000,83.500000
75%,115.25000,85.750000,92.750000
max,120.00000,95.000000,99.000000


In [21]:
df.describe(include="object")   # summary stats for text columns

,name,city,subject,exam_date
count,20,20,20,20
unique,20,5,4,4
top,Aarav Sharma,Kathmandu,Python Programming,2026-05-02
freq,1,6,6,6


`describe()` gives count, mean, std, min, quartiles, and max for numeric columns — a fast way
to sanity-check ranges (e.g. no one has more than 100 marks) before you trust the data.

## 6. Selecting columns

- `df['col']` with **single** brackets returns a **Series** (one column).
- `df[['col1', 'col2']]` with **double** brackets returns a **DataFrame** (a table, even with one column).

This distinction matters because Series and DataFrames support slightly different operations.

In [22]:
df["marks"].head()          # a Series

0    88
1    72
2    45
3    91
4    67
Name: marks, dtype: int64

In [23]:
df[["name", "marks"]].head()   # a DataFrame

,name,marks
0,Aarav Sharma,88
1,Bibek Thapa,72
2,Chandni Rai,45
3,Deepika Gurung,91
4,Esha KC,67


In [24]:
type(df["marks"]), type(df[["marks"]])

(pandas.core.series.Series, pandas.core.frame.DataFrame)

## 7. Selecting rows: `.loc` vs `.iloc`

- **`.loc`** selects by **label** — the actual index/column names. Slices with `.loc` are
  **inclusive** of the end label.
- **`.iloc`** selects by **integer position** — 0, 1, 2, ... regardless of the labels. Slices with
  `.iloc` behave like normal Python slicing: the end position is **excluded**.

Since our index here is just 0..19, the labels and positions look similar — so let's make the
difference obvious with a slice.

In [25]:
df.loc[0]     # row with index label 0

student_id                           101
name                        Aarav Sharma
city                           Kathmandu
subject               Python Programming
marks                                 88
attendance_percent                  95.0
exam_date                     2026-05-02
Name: 0, dtype: object

In [26]:
df.iloc[0]    # row at position 0 (same row here, but for a different reason)

student_id                           101
name                        Aarav Sharma
city                           Kathmandu
subject               Python Programming
marks                                 88
attendance_percent                  95.0
exam_date                     2026-05-02
Name: 0, dtype: object

In [27]:
print("loc[2:5] includes label 5:")
display(df.loc[2:5, ["name", "marks"]])

print("iloc[2:5] stops before position 5:")
display(df.iloc[2:5][["name", "marks"]])

loc[2:5] includes label 5:


,name,marks
2,Chandni Rai,45
3,Deepika Gurung,91
4,Esha KC,67
5,Faisal Khan,58


iloc[2:5] stops before position 5:


,name,marks
2,Chandni Rai,45
3,Deepika Gurung,91
4,Esha KC,67


## 8. Slicing rows and columns together

Both `.loc` and `.iloc` accept `[rows, columns]` — you can pick a block of the table in one go.

In [28]:
df.loc[0:4, ["name", "subject", "marks"]]     # rows 0-4 (inclusive), specific columns by name

,name,subject,marks
0,Aarav Sharma,Python Programming,88
1,Bibek Thapa,Statistics,72
2,Chandni Rai,Python Programming,45
3,Deepika Gurung,Machine Learning,91
4,Esha KC,Statistics,67


In [29]:
df.iloc[0:5, 1:4]      # first 5 rows, columns at positions 1,2,3

,name,city,subject
0,Aarav Sharma,Kathmandu,Python Programming
1,Bibek Thapa,Pokhara,Statistics
2,Chandni Rai,Biratnagar,Python Programming
3,Deepika Gurung,Kathmandu,Machine Learning
4,Esha KC,Lalitpur,Statistics


## 9. Boolean filtering

Writing a condition on a column (like `df['marks'] > 80`) produces a Series of `True`/`False`
values — one per row. Putting that mask inside `df[...]` keeps only the rows where it's `True`.
This is the core pattern behind almost all filtering in pandas.

In [30]:
mask = df["marks"] > 80
mask.head()      # True/False for each row

0     True
1    False
2    False
3     True
4    False
Name: marks, dtype: bool

In [31]:
df[mask].head()          # only the rows where marks > 80

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
7,108,Hari Bahadur,Kathmandu,SQL,82,90.0,2026-05-05
8,109,Isha Shrestha,Lalitpur,Statistics,95,99.0,2026-05-03
12,113,Maya Tamang,Butwal,Python Programming,90,96.0,2026-05-02


In [32]:
df[df["marks"] > 80]     # same thing, written in one line — the common style

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
7,108,Hari Bahadur,Kathmandu,SQL,82,90.0,2026-05-05
8,109,Isha Shrestha,Lalitpur,Statistics,95,99.0,2026-05-03
12,113,Maya Tamang,Butwal,Python Programming,90,96.0,2026-05-02
17,118,Rohit Bista,Kathmandu,Statistics,85,92.0,2026-05-03
19,120,Tenzin Sherpa,Lalitpur,Machine Learning,93,97.0,2026-05-04


## 10. Combining conditions

Combine multiple conditions with `&` (and), `|` (or), and `~` (not) — **not** Python's `and` / `or`,
which don't work row-by-row on a Series. Wrap each condition in parentheses.

In [33]:
# Students who scored above 80 in Python Programming
df[(df["marks"] > 80) & (df["subject"] == "Python Programming")]

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
12,113,Maya Tamang,Butwal,Python Programming,90,96.0,2026-05-02


In [34]:
# Students from Kathmandu OR Pokhara
df[(df["city"] == "Kathmandu") | (df["city"] == "Pokhara")]

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
1,102,Bibek Thapa,Pokhara,Statistics,72,88.5,2026-05-03
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
6,107,Gita Adhikari,Pokhara,Machine Learning,39,55.0,2026-05-04
7,108,Hari Bahadur,Kathmandu,SQL,82,90.0,2026-05-05
10,111,Kabita Basnet,Kathmandu,SQL,74,85.0,2026-05-05
11,112,Lokesh Poudel,Pokhara,Machine Learning,48,60.0,2026-05-04
13,114,Nabin Karki,Kathmandu,Statistics,55,75.0,2026-05-03
16,117,Queenie Rana,Pokhara,Python Programming,33,50.0,2026-05-02
17,118,Rohit Bista,Kathmandu,Statistics,85,92.0,2026-05-03


In [35]:
# Students who did NOT score above 50 (i.e. marks <= 50)
df[~(df["marks"] > 50)]

,student_id,name,city,subject,marks,attendance_percent,exam_date
2,103,Chandni Rai,Biratnagar,Python Programming,45,70.0,2026-05-02
6,107,Gita Adhikari,Pokhara,Machine Learning,39,55.0,2026-05-04
11,112,Lokesh Poudel,Pokhara,Machine Learning,48,60.0,2026-05-04
16,117,Queenie Rana,Pokhara,Python Programming,33,50.0,2026-05-02


**Common mistake:** writing `df['marks'] > 80 and df['subject'] == 'X'` raises an error, because
Python's `and`/`or` expect a single True/False, not a whole column of them. Always use `&`, `|`, `~`
with parentheses around each condition.

## 11. A more readable style: `df.query()`

`query()` takes the same condition, written as a plain string, instead of repeating `df['col']`
for every column. Many people find it easier to read, especially with several conditions.

**The rule flips here:** inside brackets you use `&` / `|` / `~`, but inside `query()` you write
`and` / `or` / `not` as plain words — the string is parsed on its own terms, not as Python code
around a Series.

In [36]:
# Same filter as before, written with query()
df.query("marks > 80 and subject == 'Python Programming'")

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
12,113,Maya Tamang,Butwal,Python Programming,90,96.0,2026-05-02


In [37]:
# OR and NOT work the same way, as plain words
df.query("city == 'Kathmandu' or city == 'Pokhara'")

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
1,102,Bibek Thapa,Pokhara,Statistics,72,88.5,2026-05-03
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
6,107,Gita Adhikari,Pokhara,Machine Learning,39,55.0,2026-05-04
7,108,Hari Bahadur,Kathmandu,SQL,82,90.0,2026-05-05
10,111,Kabita Basnet,Kathmandu,SQL,74,85.0,2026-05-05
11,112,Lokesh Poudel,Pokhara,Machine Learning,48,60.0,2026-05-04
13,114,Nabin Karki,Kathmandu,Statistics,55,75.0,2026-05-03
16,117,Queenie Rana,Pokhara,Python Programming,33,50.0,2026-05-02
17,118,Rohit Bista,Kathmandu,Statistics,85,92.0,2026-05-03


In [38]:
df.query("not marks > 50")     # marks <= 50

,student_id,name,city,subject,marks,attendance_percent,exam_date
2,103,Chandni Rai,Biratnagar,Python Programming,45,70.0,2026-05-02
6,107,Gita Adhikari,Pokhara,Machine Learning,39,55.0,2026-05-04
11,112,Lokesh Poudel,Pokhara,Machine Learning,48,60.0,2026-05-04
16,117,Queenie Rana,Pokhara,Python Programming,33,50.0,2026-05-02


Both styles do the same thing — use whichever reads more clearly for the condition at hand.
`query()` tends to win as conditions get longer; bracket filtering is often clearer for a single
simple condition.

## 12. Sorting data

`sort_values()` sorts by one or more columns; `sort_index()` sorts by the row labels.

In [39]:
df.sort_values("marks", ascending=False).head()    # highest marks first

,student_id,name,city,subject,marks,attendance_percent,exam_date
8,109,Isha Shrestha,Lalitpur,Statistics,95,99.0,2026-05-03
19,120,Tenzin Sherpa,Lalitpur,Machine Learning,93,97.0,2026-05-04
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
12,113,Maya Tamang,Butwal,Python Programming,90,96.0,2026-05-02
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02


In [40]:
# Sort by subject (A-Z), then by marks within each subject (highest first)
df.sort_values(["subject", "marks"], ascending=[True, False]).head(10)

,student_id,name,city,subject,marks,attendance_percent,exam_date
19,120,Tenzin Sherpa,Lalitpur,Machine Learning,93,97.0,2026-05-04
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
15,116,Prakash Yadav,Biratnagar,Machine Learning,77,89.0,2026-05-04
11,112,Lokesh Poudel,Pokhara,Machine Learning,48,60.0,2026-05-04
6,107,Gita Adhikari,Pokhara,Machine Learning,39,55.0,2026-05-04
12,113,Maya Tamang,Butwal,Python Programming,90,96.0,2026-05-02
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
9,110,Jeevan Lama,Biratnagar,Python Programming,61,72.0,2026-05-02
5,106,Faisal Khan,Butwal,Python Programming,58,65.0,2026-05-02
2,103,Chandni Rai,Biratnagar,Python Programming,45,70.0,2026-05-02


In [41]:
df.sort_index().head()     # back to original row order

,student_id,name,city,subject,marks,attendance_percent,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
1,102,Bibek Thapa,Pokhara,Statistics,72,88.5,2026-05-03
2,103,Chandni Rai,Biratnagar,Python Programming,45,70.0,2026-05-02
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
4,105,Esha KC,Lalitpur,Statistics,67,80.0,2026-05-03


## 13. Renaming and modifying columns

`rename(columns={...})` renames specific columns without touching the rest. By default it
returns a **new** DataFrame — the original is unchanged unless you pass `inplace=True` or
reassign the result.

In [42]:
renamed = df.rename(columns={"marks": "score", "attendance_percent": "attendance_pct"})
renamed.head()

,student_id,name,city,subject,score,attendance_pct,exam_date
0,101,Aarav Sharma,Kathmandu,Python Programming,88,95.0,2026-05-02
1,102,Bibek Thapa,Pokhara,Statistics,72,88.5,2026-05-03
2,103,Chandni Rai,Biratnagar,Python Programming,45,70.0,2026-05-02
3,104,Deepika Gurung,Kathmandu,Machine Learning,91,98.0,2026-05-04
4,105,Esha KC,Lalitpur,Statistics,67,80.0,2026-05-03


In [43]:
print(df.columns.tolist())        # unchanged — we didn't use inplace or reassign
print(renamed.columns.tolist())

['student_id', 'name', 'city', 'subject', 'marks', 'attendance_percent', 'exam_date']
['student_id', 'name', 'city', 'subject', 'score', 'attendance_pct', 'exam_date']


## 14. Adding and modifying columns

Assigning to a new column name creates it. Operations on columns are **vectorized** — they apply
to every row at once, no loop needed.

In [44]:
df["passed"] = df["marks"] >= 40      # a new boolean column
df[["name", "marks", "passed"]].head()

,name,marks,passed
0,Aarav Sharma,88,True
1,Bibek Thapa,72,True
2,Chandni Rai,45,True
3,Deepika Gurung,91,True
4,Esha KC,67,True


In [45]:
# A grade band, built from marks using np.where (nested for multiple bands)
df["grade"] = np.where(df["marks"] >= 85, "A",
              np.where(df["marks"] >= 70, "B",
              np.where(df["marks"] >= 40, "C", "F")))
df[["name", "marks", "grade"]].head(8)

,name,marks,grade
0,Aarav Sharma,88,A
1,Bibek Thapa,72,B
2,Chandni Rai,45,C
3,Deepika Gurung,91,A
4,Esha KC,67,C
5,Faisal Khan,58,C
6,Gita Adhikari,39,F
7,Hari Bahadur,82,B


## 15. Basic data type conversion

Types matter: a column stored as text can't be summed or compared numerically, and a column
stored as a generic object wastes memory if it only has a few repeated values.

In [46]:
df.dtypes

student_id              int64
name                   object
city                   object
subject                object
marks                   int64
attendance_percent    float64
exam_date              object
passed                   bool
grade                  object
dtype: object

In [47]:
# 'passed' is boolean -> convert to 0/1 integers
df["passed_flag"] = df["passed"].astype(int)
df[["passed", "passed_flag"]].head()

,passed,passed_flag
0,True,1
1,True,1
2,True,1
3,True,1
4,True,1


In [48]:
# 'subject' repeats only 5 values across 20 rows -> category dtype saves memory
before = df["subject"].memory_usage(deep=True)
df["subject"] = df["subject"].astype("category")
after = df["subject"].memory_usage(deep=True)
print(f"memory before: {before} bytes, after: {after} bytes")

memory before: 1362 bytes, after: 567 bytes


## 16. Handling dates

Right now `exam_date` is just text (an `object` dtype) — pandas doesn't know it's a date until
we tell it with `pd.to_datetime()`. Once converted, the `.dt` accessor unlocks year, month,
weekday, and more.

In [49]:
df["exam_date"].dtype     # currently just text

dtype('O')

In [50]:
df["exam_date"] = pd.to_datetime(df["exam_date"])
df["exam_date"].dtype

dtype('<M8[ns]')

In [51]:
df["exam_weekday"] = df["exam_date"].dt.day_name()
df[["name", "exam_date", "exam_weekday"]].head()

,name,exam_date,exam_weekday
0,Aarav Sharma,2026-05-02,Saturday
1,Bibek Thapa,2026-05-03,Sunday
2,Chandni Rai,2026-05-02,Saturday
3,Deepika Gurung,2026-05-04,Monday
4,Esha KC,2026-05-03,Sunday


In [52]:
# Filtering by date works just like any other condition
df[df["exam_date"] > "2026-05-03"][["name", "exam_date"]]

,name,exam_date
3,Deepika Gurung,2026-05-04
6,Gita Adhikari,2026-05-04
7,Hari Bahadur,2026-05-05
10,Kabita Basnet,2026-05-05
11,Lokesh Poudel,2026-05-04
14,Oshin Magar,2026-05-05
15,Prakash Yadav,2026-05-04
18,Sabina Ghale,2026-05-05
19,Tenzin Sherpa,2026-05-04


## Recap

- A **Series** is one labeled column; a **DataFrame** is a table of Series sharing an index.
- Always inspect a new dataset with `head()`, `shape`, `info()`, `dtypes`, and `describe()`.
- `df['col']` → Series, `df[['col']]` → DataFrame.
- `.loc` = by label (slice end included); `.iloc` = by position (slice end excluded).
- A condition on a column gives a `True`/`False` mask; `df[mask]` filters rows.
- Combine conditions with `&`, `|`, `~` — always in parentheses.
- `df.query("...")` filters the same way using a plain-English string (`and`/`or`/`not` inside it).
- `sort_values()` / `sort_index()`, `rename(columns=...)`, direct column assignment, `astype()`,
  and `pd.to_datetime()` cover most everyday cleanup work.

## Practice — try these yourself

Use the `df` DataFrame above (it now has `passed`, `grade`, `passed_flag`, and `exam_weekday`
columns from the sections above). Write your own code in the empty cells below.

1. Select only the `name` and `city` columns for students from **Lalitpur**.
2. Find all students who scored above 60 **and** have attendance above 90%.
3. Sort the students by `attendance_percent` from lowest to highest.
4. Rename the `attendance_percent` column to `attendance` (without changing the original `df`).
5. Add a new column `high_attendance` that is `True` when `attendance_percent >= 85`.

In [53]:
# 1. name and city for students from Lalitpur


In [54]:
# 2. marks > 60 and attendance_percent > 90


In [55]:
# 3. sort by attendance_percent, ascending


In [56]:
# 4. rename attendance_percent -> attendance (don't modify df)


In [57]:
# 5. add high_attendance column
